# SQL Join 与子查询 (Joins & Subqueries)

> **适用场景**: 多表关联、数据整合、层级查询、性能优化
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频

## 目录

1. 各类 JOIN 语义 & NULL 行为
2. LATERAL JOIN — 相关子查询的强大替代
3. EXISTS vs IN vs JOIN 性能对比
4. CTE (WITH) 与递归 CTE
5. Semi Join / Anti Join 优化
6. 练习题

In [ ]:
# !pip install duckdb -q
import duckdb
import pandas as pd

con = duckdb.connect()
print(f"DuckDB version: {duckdb.__version__}")

In [ ]:
# 创建示例数据
con.execute("""
CREATE OR REPLACE TABLE customers AS
SELECT * FROM (VALUES
    (1, 'Alice',   'alice@example.com',   'VIP'),
    (2, 'Bob',     'bob@example.com',     'Regular'),
    (3, 'Carol',   'carol@example.com',   'VIP'),
    (4, 'David',   'david@example.com',   'Regular'),
    (5, 'Eve',     'eve@example.com',     'VIP')
) t(customer_id, name, email, tier)
""")

con.execute("""
CREATE OR REPLACE TABLE orders AS
SELECT * FROM (VALUES
    (101, 1, '2024-01-10', 250.00),
    (102, 1, '2024-02-15', 180.00),
    (103, 2, '2024-01-20', 340.00),
    (104, 3, '2024-03-01', 520.00),
    (105, 6, '2024-03-05', 150.00)  -- customer_id=6 在 customers 中不存在
) t(order_id, customer_id, order_date, amount)
""")

# 组织架构数据（用于递归 CTE）
con.execute("""
CREATE OR REPLACE TABLE org_chart AS
SELECT * FROM (VALUES
    (1, 'CEO',           NULL, 'C-Suite'),
    (2, 'CTO',           1,    'C-Suite'),
    (3, 'CFO',           1,    'C-Suite'),
    (4, 'VP Engineering',2,    'VP'),
    (5, 'VP Product',    2,    'VP'),
    (6, 'Controller',    3,    'Director'),
    (7, 'Sr Engineer 1', 4,    'IC'),
    (8, 'Sr Engineer 2', 4,    'IC'),
    (9, 'PM Lead',       5,    'Manager'),
    (10,'Accountant',    6,    'IC')
) t(emp_id, title, manager_id, level)
""")

# 产品和评价数据（用于 LATERAL JOIN）
con.execute("""
CREATE OR REPLACE TABLE products AS
SELECT * FROM (VALUES
    (1, 'Laptop Pro',  'Electronics', 1299.99),
    (2, 'Wireless Mouse', 'Electronics', 29.99),
    (3, 'Office Chair', 'Furniture', 499.99),
    (4, 'Standing Desk', 'Furniture', 799.99)
) t(product_id, name, category, price)
""")

con.execute("""
CREATE OR REPLACE TABLE reviews AS
SELECT * FROM (VALUES
    (1, 1, 5, '2024-01-05', 'Excellent laptop!'),
    (2, 1, 4, '2024-01-12', 'Good but pricey'),
    (3, 1, 5, '2024-02-01', 'Worth every penny'),
    (4, 2, 3, '2024-01-08', 'Average quality'),
    (5, 2, 4, '2024-02-10', 'Better than expected'),
    (6, 3, 5, '2024-01-15', 'Very comfortable'),
    (7, 3, 4, '2024-02-20', 'Good ergonomics'),
    (8, 4, 2, '2024-01-22', 'Hard to assemble')
) t(review_id, product_id, rating, review_date, comment)
""")

print("所有示例数据创建完成！")
print("\n--- customers ---")
display(con.execute("SELECT * FROM customers").df())
print("\n--- orders ---")
display(con.execute("SELECT * FROM orders").df())

---

## 1. 各类 JOIN 语义 & NULL 行为

```
JOIN 类型视觉参考（韦恩图）:

INNER JOIN         LEFT JOIN          RIGHT JOIN         FULL OUTER JOIN
  A  B               A  B               A  B               A  B
 ┌─┬─┐             ┌─┬─┐             ┌─┬─┐             ┌─┬─┐
 │ │█│             │█│█│             │ │█│             │█│█│
 └─┴─┘             └─┴─┘             └─┴─┘             └─┴─┘
 交集部分           A全部+交集         B全部+交集         A+B全部
```

### NULL 行为说明
- **INNER JOIN**: 只返回两表都匹配的行，不匹配的行被丢弃
- **LEFT JOIN**: 返回左表所有行，右表无匹配时填 NULL
- **RIGHT JOIN**: 返回右表所有行，左表无匹配时填 NULL
- **FULL OUTER JOIN**: 返回两表所有行，各自无匹配时填 NULL
- **CROSS JOIN**: 笛卡尔积，返回左表×右表的所有组合

**面试陷阱**: JOIN 条件中有 NULL 时，`NULL = NULL` 为 FALSE，所以 NULL 值不会匹配！

In [ ]:
# 各种 JOIN 类型对比
print("=== INNER JOIN: 只保留两表都有匹配的行 ===")
display(con.execute("""
SELECT c.customer_id, c.name, o.order_id, o.amount
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id
""").df())
# 注意：customers 中 4(David), 5(Eve) 没有订单，被丢弃
# orders 中 order_id=105 的 customer_id=6 在 customers 中不存在，也被丢弃

print("\n=== LEFT JOIN: 保留所有客户，无订单的显示 NULL ===")
display(con.execute("""
SELECT c.customer_id, c.name, o.order_id, o.amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id
""").df())

In [ ]:
print("=== FULL OUTER JOIN: 保留两表所有行，各自无匹配填 NULL ===")
display(con.execute("""
SELECT
    c.customer_id  AS cust_customer_id,
    c.name,
    o.order_id,
    o.customer_id  AS order_customer_id,
    o.amount,
    CASE
        WHEN c.customer_id IS NULL THEN '孤儿订单（无客户记录）'
        WHEN o.order_id IS NULL    THEN '无订单客户'
        ELSE                            '正常匹配'
    END AS match_status
FROM customers c
FULL OUTER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id NULLS LAST
""").df())
# 可以看到：David(4)/Eve(5) 无订单，以及 customer_id=6 的孤儿订单

In [ ]:
# NULL 在 JOIN 中的特殊行为
con.execute("""
CREATE OR REPLACE TABLE t_null_demo AS
SELECT * FROM (VALUES
    (1, 'A'), (2, 'B'), (NULL, 'C')
) t(id, val_left)
""")

con.execute("""
CREATE OR REPLACE TABLE t_null_demo2 AS
SELECT * FROM (VALUES
    (1, 'X'), (3, 'Y'), (NULL, 'Z')
) t(id, val_right)
""")

print("NULL JOIN NULL 的行为（NULL != NULL，所以不匹配）:")
con.execute("""
SELECT l.id AS left_id, l.val_left, r.id AS right_id, r.val_right
FROM t_null_demo l
FULL OUTER JOIN t_null_demo2 r ON l.id = r.id
""").df()

---

## 2. LATERAL JOIN — 相关子查询的强大替代

`LATERAL JOIN`（也称为 `CROSS APPLY` / `OUTER APPLY`）允许子查询**引用外部表的列**，相当于对每行执行一个相关子查询。

**语法**:
```sql
SELECT *
FROM table1 t1
CROSS JOIN LATERAL (
    SELECT ... FROM table2 t2
    WHERE t2.id = t1.id  -- 引用了外部的 t1.id
    LIMIT 3
) sub
```

**经典应用**: "每个分类取最新的 N 条记录"（Top N per group）

| 方法 | 适用场景 | 性能 |
|------|----------|------|
| ROW_NUMBER + CTE | 单表 Top N | 好 |
| LATERAL JOIN | 多表关联 Top N | 通常更优 |
| 相关子查询 | 简单场景 | 一般 |

In [ ]:
# LATERAL JOIN：每个产品取最新的 2 条评价
result = con.execute("""
SELECT
    p.product_id,
    p.name AS product_name,
    p.category,
    latest_reviews.review_id,
    latest_reviews.rating,
    latest_reviews.review_date,
    latest_reviews.comment
FROM products p
CROSS JOIN LATERAL (
    -- 这个子查询可以引用外部的 p.product_id
    SELECT review_id, rating, review_date, comment
    FROM reviews r
    WHERE r.product_id = p.product_id  -- 关键：引用外表列
    ORDER BY review_date DESC
    LIMIT 2  -- 每个产品只取最新2条
) AS latest_reviews
ORDER BY p.product_id, latest_reviews.review_date DESC
""").df()

print("LATERAL JOIN：每个产品最新的 2 条评价:")
result

In [ ]:
# LATERAL JOIN 与 ROW_NUMBER 方法对比
print("方法一：使用 ROW_NUMBER + CTE")
result1 = con.execute("""
WITH ranked_reviews AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY review_date DESC
        ) AS rn
    FROM reviews
)
SELECT p.name AS product_name, r.rating, r.review_date, r.comment
FROM products p
JOIN ranked_reviews r ON p.product_id = r.product_id
WHERE r.rn <= 2
ORDER BY p.product_id, r.review_date DESC
""").df()
display(result1)

print("\n方法二：使用 LATERAL JOIN（对大表通常更高效）")
result2 = con.execute("""
SELECT p.name AS product_name, lr.rating, lr.review_date, lr.comment
FROM products p
CROSS JOIN LATERAL (
    SELECT rating, review_date, comment
    FROM reviews r
    WHERE r.product_id = p.product_id
    ORDER BY review_date DESC
    LIMIT 2
) lr
ORDER BY p.product_id, lr.review_date DESC
""").df()
display(result2)

---

## 3. EXISTS vs IN vs JOIN 性能对比

这三种模式常用于"半连接"（判断是否存在匹配），但性能和语义有差别：

### 语义差异

```sql
-- IN：将子查询结果列表化，然后检查是否包含
WHERE customer_id IN (SELECT customer_id FROM orders)

-- EXISTS：逐行检查是否有匹配，找到即停止
WHERE EXISTS (SELECT 1 FROM orders WHERE customer_id = c.customer_id)

-- JOIN（等价的 INNER JOIN）：关联后去重
SELECT DISTINCT c.* FROM customers c JOIN orders o ON c.customer_id = o.customer_id
```

### 性能规律

| 方法 | NULL 处理 | 适用场景 |
|------|-----------|----------|
| `IN` | NULL 导致不匹配（陷阱！） | 子查询结果集小，列已索引 |
| `EXISTS` | NULL 安全，找到即停止 | 大表，只需判断存在性 |
| `JOIN + DISTINCT` | 明确处理 NULL | 需要关联字段，或子查询后需聚合 |

**面试重点**: `IN` 的 NULL 陷阱！`NOT IN` 时子查询包含 NULL，结果为空集！

In [ ]:
# 三种方式实现：找出有订单的客户
print("=== 方法一：IN ===")
display(con.execute("""
SELECT customer_id, name
FROM customers
WHERE customer_id IN (
    SELECT DISTINCT customer_id FROM orders
)
""").df())

print("=== 方法二：EXISTS（推荐：找到即停止）===")
display(con.execute("""
SELECT customer_id, name
FROM customers c
WHERE EXISTS (
    SELECT 1 FROM orders o
    WHERE o.customer_id = c.customer_id
)
""").df())

print("=== 方法三：JOIN + DISTINCT ===")
display(con.execute("""
SELECT DISTINCT c.customer_id, c.name
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
""").df())

In [ ]:
# NOT IN 的 NULL 陷阱演示
con.execute("""
CREATE OR REPLACE TABLE list_with_null AS
SELECT * FROM (VALUES (1), (3), (NULL)) t(id)
""")

print("原始数据: ids = [1, 2, 3, 4, 5]")
print("子查询包含 NULL 的列表 [1, 3, NULL]")

print("\n=== NOT IN（子查询含 NULL 时返回空集！）===")
result_not_in = con.execute("""
SELECT id FROM generate_series(1, 5) t(id)
WHERE id NOT IN (SELECT id FROM list_with_null)
""").df()
print(f"结果行数: {len(result_not_in)}（期望: 2行，实际: 0行！）")
print(result_not_in)

print("\n=== NOT EXISTS（NULL 安全，正确返回）===")
result_not_exists = con.execute("""
SELECT id FROM generate_series(1, 5) t(id)
WHERE NOT EXISTS (
    SELECT 1 FROM list_with_null l
    WHERE l.id = t.id
)
""").df()
print(f"结果行数: {len(result_not_exists)}（正确返回 2 和 4 和 5）")
print(result_not_exists)

---

## 4. CTE (WITH) 与递归 CTE

### 普通 CTE

CTE（Common Table Expression）使用 `WITH` 子句定义命名的临时结果集，提高可读性并允许复用。

```sql
WITH cte_name AS (
    SELECT ...
),
another_cte AS (
    SELECT ... FROM cte_name  -- 可以引用之前定义的 CTE
)
SELECT * FROM another_cte;
```

### 递归 CTE

```sql
WITH RECURSIVE rec_cte AS (
    -- 锚点部分（初始值）
    SELECT ... -- 非递归部分
    UNION ALL
    -- 递归部分（引用自身）
    SELECT ... FROM rec_cte WHERE ...
)
SELECT * FROM rec_cte;
```

**典型应用**:
- 遍历层级结构（组织架构、分类树）
- 生成日期序列
- 路径查找

In [ ]:
# 普通 CTE：链式数据转换
result = con.execute("""
WITH
-- 第一步：计算每个客户的总订单金额
customer_spending AS (
    SELECT
        customer_id,
        SUM(amount)  AS total_spent,
        COUNT(*)     AS order_count
    FROM orders
    GROUP BY customer_id
),
-- 第二步：关联客户信息并分层
customer_with_tier AS (
    SELECT
        c.name,
        c.tier,
        cs.total_spent,
        cs.order_count,
        CASE
            WHEN cs.total_spent >= 500 THEN 'High Value'
            WHEN cs.total_spent >= 200 THEN 'Mid Value'
            ELSE                            'Low Value'
        END AS value_segment
    FROM customers c
    LEFT JOIN customer_spending cs ON c.customer_id = cs.customer_id
),
-- 第三步：汇总统计
segment_summary AS (
    SELECT
        value_segment,
        COUNT(*) AS customer_count,
        ROUND(AVG(total_spent), 2) AS avg_spending
    FROM customer_with_tier
    WHERE total_spent IS NOT NULL
    GROUP BY value_segment
)
SELECT * FROM segment_summary ORDER BY avg_spending DESC
""").df()

print("链式 CTE：客户价值分层统计")
result

In [ ]:
# 递归 CTE：遍历组织架构树
result = con.execute("""
WITH RECURSIVE org_tree AS (
    -- 锚点：从 CEO（manager_id IS NULL）开始
    SELECT
        emp_id,
        title,
        manager_id,
        level,
        0            AS depth,          -- 层级深度
        title        AS path,            -- 组织路径
        CAST(emp_id AS VARCHAR) AS id_path
    FROM org_chart
    WHERE manager_id IS NULL

    UNION ALL

    -- 递归：找每个节点的直接下属
    SELECT
        oc.emp_id,
        oc.title,
        oc.manager_id,
        oc.level,
        ot.depth + 1,
        ot.path || ' > ' || oc.title,      -- 构建路径字符串
        ot.id_path || '.' || CAST(oc.emp_id AS VARCHAR)
    FROM org_chart oc
    INNER JOIN org_tree ot ON oc.manager_id = ot.emp_id  -- 与递归结果关联
)
SELECT
    emp_id,
    REPEAT('  ', depth) || title AS indented_title,  -- 缩进显示层级
    depth,
    path,
    level
FROM org_tree
ORDER BY id_path
""").df()

print("递归 CTE：组织架构树遍历")
result

In [ ]:
# 递归 CTE：生成日期序列（实用技巧）
result = con.execute("""
WITH RECURSIVE date_series AS (
    SELECT DATE '2024-01-01' AS dt
    UNION ALL
    SELECT dt + INTERVAL '1 day'
    FROM date_series
    WHERE dt < DATE '2024-01-07'  -- 生成7天
)
SELECT
    dt AS date,
    DAYNAME(dt) AS day_of_week
FROM date_series
""").df()

print("递归 CTE：生成日期序列")
result

---

## 5. Semi Join / Anti Join 优化

### Semi Join（半连接）
返回左表中**在右表中有匹配的行**，但不返回右表的列，也不因右表多行而膨胀。

```sql
-- Semi Join（找有订单的客户，不需要订单详情）
SELECT * FROM customers c
WHERE EXISTS (SELECT 1 FROM orders WHERE customer_id = c.customer_id)
```

### Anti Join（反连接）
返回左表中**在右表中没有匹配的行**。

```sql
-- Anti Join（找没有订单的客户）
SELECT * FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders WHERE customer_id = c.customer_id)
-- 或
SELECT c.* FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL
```

### 为什么重要？
优化器会识别 Semi/Anti Join 模式并使用专用算法（Hash Semi Join / Merge Anti Join），避免结果集膨胀，通常比显式 JOIN 后 DISTINCT 更高效。

In [ ]:
# Semi Join vs Anti Join 对比
print("=== Semi Join：有订单的客户（EXISTS）===")
display(con.execute("""
SELECT c.customer_id, c.name, c.tier
FROM customers c
WHERE EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
)
""").df())

print("\n=== Anti Join：没有订单的客户（NOT EXISTS）===")
display(con.execute("""
SELECT c.customer_id, c.name, c.tier
FROM customers c
WHERE NOT EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
)
""").df())

print("\n=== Anti Join 等价写法：LEFT JOIN + IS NULL ===")
display(con.execute("""
SELECT c.customer_id, c.name, c.tier
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL  -- 关键：过滤掉成功匹配的行
""").df())

In [ ]:
# 查看执行计划，确认优化器识别了 Semi Join
print("EXISTS 的执行计划（Semi Join）:")
plan = con.execute("""
EXPLAIN
SELECT c.customer_id, c.name
FROM customers c
WHERE EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
)
""").fetchall()
for row in plan:
    print(row[0])

---

## 练习题

In [ ]:
# 练习数据准备
con.execute("""
CREATE OR REPLACE TABLE departments AS
SELECT * FROM (VALUES
    (10, 'Engineering'),
    (20, 'Marketing'),
    (30, 'HR'),
    (40, 'Finance')  -- 没有员工的部门
) t(dept_id, dept_name)
""")

con.execute("""
CREATE OR REPLACE TABLE emp_join AS
SELECT * FROM (VALUES
    (1, 'Alice',   10, 95000, DATE '2020-03-01'),
    (2, 'Bob',     10, 88000, DATE '2021-06-15'),
    (3, 'Carol',   20, 72000, DATE '2022-08-20'),
    (4, 'David',   20, 68000, DATE '2021-11-30'),
    (5, 'Eve',     30, 58000, DATE '2023-01-05'),
    (6, 'Frank',   NULL, 0, DATE '2023-06-01')  -- 未分配部门
) t(emp_id, name, dept_id, salary, hire_date)
""")

con.execute("""
CREATE OR REPLACE TABLE projects AS
SELECT * FROM (VALUES
    (101, 'Project Alpha', 10, '2024-01-01', '2024-06-30'),
    (102, 'Project Beta',  20, '2024-02-01', '2024-08-31'),
    (103, 'Project Gamma', 10, '2024-03-01', '2024-12-31'),
    (104, 'Project Delta', 30, '2024-04-01', '2024-10-31')
) t(proj_id, proj_name, dept_id, start_date, end_date)
""")

print("练习数据准备完成！")
display(con.execute("SELECT * FROM departments").df())
display(con.execute("SELECT * FROM emp_join").df())

### 练习 1: JOIN 类型选择

**需求**: 生成一份完整的部门员工报告，要求：
- 显示所有部门（包括没有员工的 Finance 部门）
- 显示所有员工（包括未分配部门的 Frank）
- 对没有员工的部门，员工相关列显示为 NULL
- 对没有部门的员工，部门名显示为 '未分配'

**提示**: 使用 FULL OUTER JOIN + COALESCE

In [ ]:
# 练习 1: JOIN 类型选择（填写 TODO）
result = con.execute("""
SELECT
    COALESCE(d.dept_name, '未分配')  AS dept_name,
    e.name                            AS emp_name,
    e.salary
FROM departments d
-- TODO: 选择正确的 JOIN 类型
TODO JOIN emp_join e ON d.dept_id = e.dept_id
ORDER BY d.dept_name NULLS LAST, e.name
""").df()
result

In [ ]:
# 练习 1 参考答案
result = con.execute("""
SELECT
    COALESCE(d.dept_name, '未分配') AS dept_name,
    e.name                           AS emp_name,
    e.salary
FROM departments d
FULL OUTER JOIN emp_join e ON d.dept_id = e.dept_id
ORDER BY d.dept_name NULLS LAST, e.name
""").df()
result

### 练习 2: 递归 CTE — 查找管理链

**需求**: 使用 org_chart 表，找出 CEO 到每个员工的完整管理链路，并显示层级深度。

**要求**:
1. 使用递归 CTE 从根节点（CEO）向下遍历
2. 显示每个员工的完整路径（如: CEO > CTO > VP Engineering > Sr Engineer 1）
3. 显示员工的管理层级深度（CEO = 0）

In [ ]:
# 练习 2: 递归 CTE（填写 TODO）
result = con.execute("""
WITH RECURSIVE management_chain AS (
    -- TODO: 锚点部分（顶层管理者）
    SELECT
        emp_id,
        title,
        manager_id,
        0 AS depth,
        title AS full_path
    FROM org_chart
    WHERE TODO  -- 从没有上级的员工开始

    UNION ALL

    -- TODO: 递归部分
    SELECT
        oc.emp_id,
        oc.title,
        oc.manager_id,
        mc.depth + 1,
        mc.full_path || ' > ' || oc.title
    FROM org_chart oc
    TODO JOIN management_chain mc ON TODO  -- 关联条件
)
SELECT
    emp_id,
    REPEAT('  ', depth) || title AS indented_title,
    depth,
    full_path
FROM management_chain
ORDER BY full_path
""").df()
result

In [ ]:
# 练习 2 参考答案
result = con.execute("""
WITH RECURSIVE management_chain AS (
    SELECT
        emp_id,
        title,
        manager_id,
        0 AS depth,
        title AS full_path
    FROM org_chart
    WHERE manager_id IS NULL

    UNION ALL

    SELECT
        oc.emp_id,
        oc.title,
        oc.manager_id,
        mc.depth + 1,
        mc.full_path || ' > ' || oc.title
    FROM org_chart oc
    INNER JOIN management_chain mc ON oc.manager_id = mc.emp_id
)
SELECT
    emp_id,
    REPEAT('  ', depth) || title AS indented_title,
    depth,
    full_path
FROM management_chain
ORDER BY full_path
""").df()
result

### 练习 3: Anti Join — 找出异常数据

**需求**:
1. 找出所有**没有参与任何项目**的部门
2. 找出所有**没有分配到任何部门**的员工

**要求**: 分别用 `NOT EXISTS` 和 `LEFT JOIN + IS NULL` 两种方式实现，结果一致。

In [ ]:
# 练习 3: Anti Join（填写 TODO）

print("问题 1: 没有项目的部门")
# 方法 A: NOT EXISTS
result_a = con.execute("""
SELECT dept_id, dept_name
FROM departments d
WHERE NOT EXISTS (
    SELECT 1 FROM projects p WHERE TODO  -- 关联条件
)
""").df()
display(result_a)

# 方法 B: LEFT JOIN + IS NULL
result_b = con.execute("""
SELECT d.dept_id, d.dept_name
FROM departments d
TODO JOIN projects p ON d.dept_id = p.dept_id
WHERE TODO  -- 过滤无匹配行
""").df()
display(result_b)

print("\n问题 2: 没有分配部门的员工")
result_c = con.execute("""
SELECT emp_id, name
FROM emp_join
WHERE TODO  -- dept_id 为 NULL 即未分配
""").df()
display(result_c)

In [ ]:
# 练习 3 参考答案
print("问题 1 - 方法 A: NOT EXISTS")
display(con.execute("""
SELECT dept_id, dept_name
FROM departments d
WHERE NOT EXISTS (
    SELECT 1 FROM projects p WHERE p.dept_id = d.dept_id
)
""").df())

print("问题 1 - 方法 B: LEFT JOIN + IS NULL")
display(con.execute("""
SELECT d.dept_id, d.dept_name
FROM departments d
LEFT JOIN projects p ON d.dept_id = p.dept_id
WHERE p.dept_id IS NULL
""").df())

print("问题 2: 未分配部门的员工")
display(con.execute("""
SELECT emp_id, name
FROM emp_join
WHERE dept_id IS NULL
""").df())

### 练习 4: LATERAL JOIN — 每部门 Top 1 薪资员工

**需求**: 使用 LATERAL JOIN 找出每个部门薪资最高的员工，并同时显示部门的项目信息。

**输出列**: dept_name, top_earner_name, top_salary, dept_projects（部门项目数量）

In [ ]:
# 练习 4: LATERAL JOIN（填写 TODO）
result = con.execute("""
SELECT
    d.dept_name,
    top_emp.name     AS top_earner_name,
    top_emp.salary   AS top_salary,
    dept_proj.project_count
FROM departments d
-- TODO: 使用 LATERAL JOIN 获取每个部门薪资最高的员工
CROSS JOIN LATERAL (
    SELECT name, salary
    FROM emp_join e
    WHERE e.dept_id = d.dept_id
    ORDER BY TODO
    LIMIT 1
) AS top_emp
-- TODO: 再 JOIN 项目计数
LEFT JOIN LATERAL (
    SELECT COUNT(*) AS project_count
    FROM projects p
    WHERE TODO
) AS dept_proj ON TRUE
ORDER BY top_emp.salary DESC
""").df()
result

In [ ]:
# 练习 4 参考答案
result = con.execute("""
SELECT
    d.dept_name,
    top_emp.name     AS top_earner_name,
    top_emp.salary   AS top_salary,
    dept_proj.project_count
FROM departments d
CROSS JOIN LATERAL (
    SELECT name, salary
    FROM emp_join e
    WHERE e.dept_id = d.dept_id
    ORDER BY salary DESC
    LIMIT 1
) AS top_emp
LEFT JOIN LATERAL (
    SELECT COUNT(*) AS project_count
    FROM projects p
    WHERE p.dept_id = d.dept_id
) AS dept_proj ON TRUE
ORDER BY top_emp.salary DESC
""").df()
result

### 练习 5: 综合题 — 自关联 + CTE + 窗口函数

**需求**: 找出每个部门中，薪资高于本部门平均薪资的员工，并显示他们高出多少百分比。

**要求**: 使用 CTE 先计算部门平均薪资，再 JOIN 过滤。

In [ ]:
# 练习 5: 综合题（填写 TODO）
result = con.execute("""
WITH dept_avg AS (
    -- TODO: 计算每个部门的平均薪资
    SELECT
        dept_id,
        ROUND(AVG(salary), 2) AS avg_salary
    FROM emp_join
    WHERE dept_id IS NOT NULL
    GROUP BY TODO
)
SELECT
    d.dept_name,
    e.name,
    e.salary,
    da.avg_salary AS dept_avg_salary,
    -- TODO: 计算超出部门均值的百分比
    ROUND((e.salary - da.avg_salary) / da.avg_salary * 100, 2) AS pct_above_avg
FROM emp_join e
JOIN TODO ON e.dept_id = da.dept_id       -- 关联部门均值
JOIN departments d ON e.dept_id = d.dept_id
WHERE TODO                                 -- 过滤：只保留高于均值的员工
ORDER BY d.dept_name, pct_above_avg DESC
""").df()
result

In [ ]:
# 练习 5 参考答案
result = con.execute("""
WITH dept_avg AS (
    SELECT
        dept_id,
        ROUND(AVG(salary), 2) AS avg_salary
    FROM emp_join
    WHERE dept_id IS NOT NULL
    GROUP BY dept_id
)
SELECT
    d.dept_name,
    e.name,
    e.salary,
    da.avg_salary AS dept_avg_salary,
    ROUND((e.salary - da.avg_salary) / da.avg_salary * 100, 2) AS pct_above_avg
FROM emp_join e
JOIN dept_avg da ON e.dept_id = da.dept_id
JOIN departments d ON e.dept_id = d.dept_id
WHERE e.salary > da.avg_salary
ORDER BY d.dept_name, pct_above_avg DESC
""").df()
result

---

## 复习要点

### JOIN 类型速查

| JOIN 类型 | 保留行 | 用途 |
|-----------|--------|------|
| INNER JOIN | 两表都匹配 | 正常关联 |
| LEFT JOIN | 左表全部 | 左表为主，右表可选 |
| RIGHT JOIN | 右表全部 | 右表为主（可改写为 LEFT JOIN） |
| FULL OUTER JOIN | 两表全部 | 数据整合，查找不匹配行 |
| CROSS JOIN | 笛卡尔积 | 生成所有组合 |
| LATERAL JOIN | 相关子查询 | Top N per group |

### 关键陷阱

1. **NOT IN + NULL 陷阱**: 子查询返回包含 NULL 的列表时，NOT IN 返回空结果。始终用 NOT EXISTS 代替
2. **LAST_VALUE 陷阱**: 需要指定完整窗口帧
3. **FULL JOIN 后的过滤**: 过滤一侧 NULL 等价于 INNER JOIN，需注意
4. **递归 CTE 无限循环**: 必须有终止条件，或用 MAXRECURSION 限制

### 面试高频问题

- "INNER JOIN vs LEFT JOIN 的区别" → 保留行的范围
- "EXISTS vs IN 哪个性能更好" → 取决于场景，NOT IN 有 NULL 陷阱
- "如何实现 Top N per group" → ROW_NUMBER + CTE 或 LATERAL JOIN
- "如何查询组织架构树" → 递归 CTE
- "什么是 Semi Join/Anti Join" → 半连接/反连接，EXISTS/NOT EXISTS 的优化形态